# Integrate all cases RNA-seq

### 🎯 **Gene Type Selection for OmicsML Project**  

Since My goal is to:  
✅ **Classify cancer subtypes**  
✅ **Identify key biomarkers**  
✅ **Predict potential drug targets**  

My **keep genes** should be biologically relevant for cancer, biomarkers, and drug targets while **removing genes** that are unlikely to contribute meaningful information.  

---

## **🛠️ Recommended Gene Types to Keep**  

| **Category**               | **Gene Types**                                               | **Keep?**            | **Reason**  |
|----------------------------|------------------------------------------------------------|----------------------|-------------|
| **Protein-Coding Genes**    | `protein_coding`                                          | ✅ **Yes** (Highly Recommended) | Essential for cancer biology & drug target discovery. |
| **Long Non-Coding RNA (lncRNA)** | `lncRNA`                                          | ✅ **Yes** (Recommended) | Regulate gene expression, impact cancer progression, & serve as biomarkers. |
| **Mitochondrial RNA (mtRNA)** | `Mt_tRNA`, `Mt_rRNA`                                 | ⚠️ **Maybe** | If studying metabolism-related pathways or drug resistance, keep these. Otherwise, remove them. |
| **Immune-Related Genes**    | `IG_V_gene`, `IG_C_gene`, `IG_D_gene`, `TR_V_gene`, `TR_J_gene`, `TR_C_gene`, `TR_D_gene` | ⚠️ **Maybe** | If your study involves tumor microenvironment or immunotherapy, keep these. Otherwise, remove them. |

---

## **🗑️ Gene Types to Remove**  

| **Category**                | **Gene Types**                                           | **Remove?**         | **Reason**  |
|-----------------------------|--------------------------------------------------------|---------------------|-------------|
| **Pseudogenes**             | `transcribed_unitary_pseudogene`, `processed_pseudogene`, `unprocessed_pseudogene`, etc. | ❌ **Remove** | Non-functional copies of genes; do not code for proteins (except in rare cases). |
| **Small Non-Coding RNAs**    | `miRNA`, `snRNA`, `snoRNA`, `scRNA`, `scaRNA`, `vault_RNA`, `ribozyme`, `sRNA` | ❌ **Remove** | Important for regulation, but not relevant for proteomics-based drug discovery. |
| **Ribosomal RNA (rRNA)**     | `rRNA`, `Mt_rRNA`, `rRNA_pseudogene`                   | ❌ **Remove** | Highly abundant, non-informative for subtype classification or drug targets. |
| **TEC (To be Experimentally Confirmed)** | `TEC`                                    | ❌ **Remove** | Not well-characterized; usually not relevant for biomarker discovery. |

For cancer subtyping, biomarker discovery, and drug target prediction, we should use unstranded counts, unless you specifically need strand information for non-coding RNA studies.

In [ ]:
import pandas as pd
import os

def process_rna_seq_file(file_path, case_id, keep_gene_types):
    """
    Process an RNA-Seq file, filter relevant gene types, and rename the expression column using the case ID.

    Parameters:
    - file_path (str): Path to the RNA-Seq file.
    - case_id (str): Case ID to rename the "unstranded" column.
    - keep_gene_types (list): List of gene types to keep in the dataset.
    
    Returns:
    - pd.DataFrame: Processed RNA-Seq data with 'gene_id' and renamed expression column.
    """

    # Define column names
    column_names = ['gene_id', 'gene_name', 'gene_type', 'unstranded', 
                    'stranded_first', 'stranded_second', 'tpm_unstranded', 
                    'fpkm_unstranded', 'fpkm_uq_unstranded']

    # Read the TSV file
    rna_seq_df = pd.read_csv(file_path, sep='\t', names=column_names, skiprows=1)
    rna_seq_df = rna_seq_df[5:]  # Skip first 5 rows (QC metrics)

    # Filter dataset based on gene type
    filtered_rna_data = rna_seq_df[rna_seq_df["gene_type"].isin(keep_gene_types)]

    # Keep only relevant columns and rename "unstranded" column to the case ID
    filtered_rna_data = filtered_rna_data[['gene_name', 'unstranded']].rename(columns={"unstranded": case_id})

    return filtered_rna_data


In [ ]:
import pandas as pd
import os

def merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, output_path):
    """
    Process multiple RNA-Seq files, filter by gene types, and merge them into a single DataFrame.

    Parameters:
        rna_seq_files (pd.DataFrame): DataFrame containing "Case ID" and "File Name".
        base_dir (str): Base directory where RNA-Seq files are stored.
        keep_gene_types (list): List of gene types to keep.
        output_name (str): Name of the output CSV file.

    Returns:
        pd.DataFrame: Combined and aligned RNA-Seq DataFrame.
    """
    combined_df = pd.DataFrame()  # Initialize an empty DataFrame

    for index, row in rna_seq_files.iterrows():
        case_id = row["Case ID"]
        file_name = row["File Name"]

        # Construct full file path
        file_path = os.path.join(base_dir, file_name)  

        # Process RNA-Seq file
        processed_df = process_rna_seq_file(file_path, case_id, keep_gene_types)
        processed_df = processed_df.groupby("gene_name").sum().reset_index() # Sum expression values for duplicate gene names

        # Transpose and format DataFrame
        processed_df = processed_df.T
        processed_df.columns = processed_df.iloc[0]  # Set column names to gene IDs
        processed_df = processed_df[1:]  # Remove the old header row
        processed_df.columns.name = 'case_id/gene_name'  # Set index name

        # Merge with combined DataFrame (align columns, fill missing values with 0)
        combined_df = pd.concat([combined_df, processed_df], axis=0, join="outer", ignore_index=False)

    # Fill missing values with 0
    combined_df = combined_df.fillna(0)

    # Save the final merged DataFrame
    combined_df.to_csv(output_path, index=True)

    print(f"✅ RNA-Seq data merged and saved as: {output_path}")

    return combined_df  # Return the merged DataFrame

In [ ]:
import pandas as pd
import os

# Define base directory for RNA-Seq files
base_dir = "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/1. Data_preparation-matching_ID/RNA-seq/"

# Load the mapping file
rna_seq_files = pd.read_csv(os.path.join(base_dir, "rna_seq_file_mapping.csv"))  

# Initialize an empty DataFrame for combined data
combined_df = pd.DataFrame()
# Define gene types to KEEP (biologically relevant)

keep_gene_types = ["protein_coding", "lncRNA", "IG_V_gene", 
                       "TR_V_gene", "TR_J_gene", "TR_C_gene", "TR_D_gene"]

merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/combined_rna_seq.csv")


✅ RNA-Seq data merged and saved as: /Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/combined_rna_seq.csv


case_id/gene_name,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2ML1-AS2,A3GALT2,A4GALT,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423
22BR005,10,30,23,13990,25,53,0,0,1,340,...,120,545,729,195,801,1114,1216,2792,0,0
05BR001,7,34,5,21575,29,159,1,0,2,170,...,174,509,533,53,1039,3349,1334,1150,0,0
18BR002,18,75,1,14006,67,29,0,0,2,291,...,110,347,1118,188,998,3126,1268,1280,0,0
11BR018,12,53,1,26543,88,23,0,0,1,275,...,165,479,799,44,1030,2633,1829,1568,0,0
11BR022,2,40,4,13618,32,19,0,0,0,210,...,202,1189,913,133,790,2511,1389,972,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
06BR014,23,150,10,8910,14,5,0,0,4,119,...,157,651,1081,284,1343,2903,2304,1654,0,0
01BR027,3,8,0,27417,80,5742,0,0,60,57,...,87,467,836,639,928,1965,908,2196,0,0
11BR025,8,45,0,6585,48,18,0,0,1,101,...,240,902,1169,13,1035,991,3562,1283,0,0
11BR036,7,52,0,4173,23,20,0,0,3,133,...,30,154,1588,137,1247,561,1882,1607,0,0


In [ ]:
keep_gene_types = ["protein_coding","lncRNA"]

merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/prot+lncRNA.csv")


✅ RNA-Seq data merged and saved as: /Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/prot+lncRNA.csv


case_id/gene_name,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2ML1-AS2,A3GALT2,A4GALT,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423
22BR005,10,30,23,13990,25,53,0,0,1,340,...,120,545,729,195,801,1114,1216,2792,0,0
05BR001,7,34,5,21575,29,159,1,0,2,170,...,174,509,533,53,1039,3349,1334,1150,0,0
18BR002,18,75,1,14006,67,29,0,0,2,291,...,110,347,1118,188,998,3126,1268,1280,0,0
11BR018,12,53,1,26543,88,23,0,0,1,275,...,165,479,799,44,1030,2633,1829,1568,0,0
11BR022,2,40,4,13618,32,19,0,0,0,210,...,202,1189,913,133,790,2511,1389,972,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
06BR014,23,150,10,8910,14,5,0,0,4,119,...,157,651,1081,284,1343,2903,2304,1654,0,0
01BR027,3,8,0,27417,80,5742,0,0,60,57,...,87,467,836,639,928,1965,908,2196,0,0
11BR025,8,45,0,6585,48,18,0,0,1,101,...,240,902,1169,13,1035,991,3562,1283,0,0
11BR036,7,52,0,4173,23,20,0,0,3,133,...,30,154,1588,137,1247,561,1882,1607,0,0


In [ ]:
keep_gene_types = ["protein_coding"]

merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/protein_coding.csv")


✅ RNA-Seq data merged and saved as: /Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/protein_coding.csv


case_id/gene_name,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
22BR005,10,23,13990,53,1,340,1,963,969,8,...,507,1697,120,545,729,195,801,1114,1216,2792
05BR001,7,5,21575,159,2,170,2,767,401,6,...,459,1070,174,509,533,53,1039,3349,1334,1150
18BR002,18,1,14006,29,2,291,6,1380,675,453,...,678,1032,110,347,1118,188,998,3126,1268,1280
11BR018,12,1,26543,23,1,275,3,1022,975,71,...,383,357,165,479,799,44,1030,2633,1829,1568
11BR022,2,4,13618,19,0,210,2,1253,1299,0,...,500,1311,202,1189,913,133,790,2511,1389,972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
06BR014,23,10,8910,5,4,119,4,1509,1735,0,...,492,1086,157,651,1081,284,1343,2903,2304,1654
01BR027,3,0,27417,5742,60,57,5,835,904,0,...,814,769,87,467,836,639,928,1965,908,2196
11BR025,8,0,6585,18,1,101,1,1562,1285,0,...,413,1246,240,902,1169,13,1035,991,3562,1283
11BR036,7,0,4173,20,3,133,5,748,2645,84,...,646,4679,30,154,1588,137,1247,561,1882,1607


In [ ]:
keep_gene_types = ["lncRNA"]

merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/lncRNA.csv")


✅ RNA-Seq data merged and saved as: /Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/lncRNA.csv


case_id/gene_name,A1BG-AS1,A2M-AS1,A2ML1-AS1,A2ML1-AS2,AA06,AADACL2-AS1,AATBC,ABALON,ABCA9-AS1,ABCC5-AS1,...,ZNNT1,ZNRD2-AS1,ZNRF3-AS1,ZNRF3-IT1,ZRANB2-AS1,ZRANB2-AS2,ZSCAN16-AS1,ZSWIM8-AS1,hsa-mir-1253,hsa-mir-423
22BR005,30,25,0,0,0,3,32,0,1,0,...,143,71,1,1,4,21,60,1,0,0
05BR001,34,29,1,0,0,0,28,0,10,1,...,145,67,1,1,4,15,69,0,0,0
18BR002,75,67,0,0,0,0,71,2,1,0,...,98,104,2,0,6,28,103,0,0,0
11BR018,53,88,0,0,0,8,52,0,9,0,...,146,75,0,0,9,34,173,0,0,0
11BR022,40,32,0,0,0,6,123,1,0,3,...,73,103,0,0,5,22,79,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
06BR014,150,14,0,0,0,4,111,0,3,0,...,46,86,0,0,4,24,72,3,0,0
01BR027,8,80,0,0,0,0,147,1,0,3,...,73,85,2,1,2,9,67,0,0,0
11BR025,45,48,0,0,0,0,12,0,0,0,...,510,141,2,0,5,66,242,0,0,0
11BR036,52,23,0,0,0,4,1592,0,2,1,...,91,89,0,0,4,36,274,0,0,0


In [ ]:

keep_gene_types = ["IG_V_gene", 
                       "TR_V_gene", "TR_J_gene", "TR_C_gene", "TR_D_gene"]

merge_rna_seq(rna_seq_files, base_dir, keep_gene_types, "/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/Immune.csv")


✅ RNA-Seq data merged and saved as: /Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/RNA-seq/Immune.csv


case_id/gene_name,AC135068.3,AC135068.9,IGHV1-18,IGHV1-2,IGHV1-24,IGHV1-3,IGHV1-45,IGHV1-46,IGHV1-58,IGHV1-69,...,TRGJP2,TRGV1,TRGV10,TRGV11,TRGV2,TRGV3,TRGV4,TRGV5,TRGV8,TRGV9
22BR005,3,0,935,159,43,296,8,163,63,14,...,0,2,11,0,11,12,11,6,3,4
05BR001,1,0,1179,610,2208,3031,2,885,65,385,...,4,7,19,1,9,16,17,5,7,24
18BR002,0,0,69,29,28,84,17,48,2,60,...,0,3,17,0,9,10,11,7,4,2
11BR018,3,0,1690,506,1353,1768,34,1227,223,380,...,2,3,24,0,17,29,22,15,5,10
11BR022,4,0,1479,2185,476,729,61,1275,81,286,...,0,0,8,0,4,6,6,3,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
06BR014,0,0,35,74,12,0,1,13,2,13,...,0,3,3,0,1,3,4,4,3,3
01BR027,2,0,17114,170,48,27,3,51,7,14,...,0,3,5,0,2,2,8,6,0,0
11BR025,0,0,9,7,4,5,0,5,0,1,...,0,1,1,0,1,6,2,2,2,1
11BR036,0,0,3,0,0,2,0,5,0,3,...,0,0,0,0,1,3,2,1,1,0


# Protemoics integration

In [128]:
prot_files = pd.read_csv("/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/1. Data_preparation-matching_ID/Proteome/Proteomic_log_merged.csv")
                           
prot_files = prot_files.T
prot_files.columns = prot_files.iloc[0]  # Set column names to gene IDs
prot_files = prot_files[1:]  # Remove the old header row
prot_files.columns.name = 'case_id/gene_name'  # Set index name
prot_files = prot_files.fillna(0)

print(prot_files)

case_id/gene_name                      A1BG       A2M     A2ML1      AAAS  \
01f203fd-e664-4d78-bea0-7c40cf_D2  0.600460  0.018012 -1.729258 -0.104503   
071be0fb-b7fd-4165-b3c5-348352    -0.972301 -0.750736  0.883655  0.571254   
079b5600-6afc-4785-bb22-48cfab_D2 -0.388622 -0.414193 -0.819298  0.402782   
09659708-7747-4d59-a3b9-e221e0_D2  0.537899  0.911622  0.178475  0.270463   
0bb9d596-774e-452b-9c89-a6643c_D2  0.240436  0.457311 -1.019009  0.233503   
...                                     ...       ...       ...       ...   
faadb553-9437-42cb-b469-498d67_D2  0.157149 -0.294779 -1.490255  0.442469   
fb526979-ee2d-4edf-b41d-01074f_D2  0.160579 -0.027269 -1.558988 -0.597002   
fc0aa1e4-c1d8-43ed-a5cd-038f61_D2  0.008241  0.180106 -1.360989  0.043123   
fe591474-99d7-44d3-be73-c3b8ed_D2  1.183481  1.285099 -1.406133 -0.182695   
ffc76bfb-61f6-4492-a782-c23321_D2  0.133543  0.065744 -0.822996 -0.020953   

case_id/gene_name                      AACS     AADAT     AAGAB      AAK1  

/var/folders/7g/j5s0yzcj34l3v043s7znkplc0000gn/T/ipykernel_55760/3340890506.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prot_files = prot_files.fillna(0)


In [136]:
matched_files = pd.read_csv("/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/1. Data_preparation-matching_ID/Proteome/matched_samples_with_aliquot.csv") 

# Create a mapping from "Aliquot Submitter ID" to "Case Submitter ID"
aliquot_to_case = dict(zip(matched_files["Aliquot Submitter ID"], matched_files["Case Submitter ID"]))

# Rename index in prot_files using the mapping
prot_files = prot_files.rename(index=aliquot_to_case)

# Display result
print(prot_files)

combined_df.to_csv("/Users/deweywang/Desktop/GitHub/OmicsFlow/1. OmicsML/dataset/processed/2. RNA_seq cleaning/Proteome/Proteome.csv", index=True)



case_id/gene_name      A1BG       A2M     A2ML1      AAAS      AACS     AADAT  \
03BR013            0.600460  0.018012 -1.729258 -0.104503 -0.558639 -0.236499   
03BR006           -0.972301 -0.750736  0.883655  0.571254 -0.522824  0.000000   
11BR023           -0.388622 -0.414193 -0.819298  0.402782 -0.528798  0.000000   
01BR017            0.537899  0.911622  0.178475  0.270463 -0.014834  0.000000   
18BR010            0.240436  0.457311 -1.019009  0.233503 -0.242060  0.000000   
...                     ...       ...       ...       ...       ...       ...   
11BR013            0.157149 -0.294779 -1.490255  0.442469 -0.650083 -1.098859   
11BR053            0.160579 -0.027269 -1.558988 -0.597002 -0.352398  0.506232   
11BR059            0.008241  0.180106 -1.360989  0.043123 -0.251022  0.458397   
18BR002            1.183481  1.285099 -1.406133 -0.182695 -0.814892  0.313926   
05BR003            0.133543  0.065744 -0.822996 -0.020953 -0.353886  0.000000   

case_id/gene_name     AAGAB